# App concept: Content Promotion Planner

**Purpose**: To help a marketing team answer: 

*Given a fixed budget and safety constraints, how many posts should we promote, what needs review, and what return might we expect?*

This app turns the notebook decision system into an interactive planning tool. Rather than only asking “which posts should be promoted?”, it lets users explore how promotion volume, moderation cost, safety thresholds and expected value affect the final decision.

The calculations are directional rather than predictive finance: engagement probabilities are used as a proxy for expected performance, and ROI depends on user-defined assumptions.

⚠️ Note: For this prototype, model scoring is precomputed. The app focuses on the decision layer: how different budget, safety and value assumptions change promotion outcomes.

In [1]:
import pandas as pd

In [2]:
# Load decision system dataframe
decision_system_df = pd.read_parquet("../data/processed/decision_system_df.parquet")
print("✅ Decision System dataframe loaded from ../data/processed/decision_system_df.parquet")
decision_system_df.head()

✅ Decision System dataframe loaded from ../data/processed/decision_system_df.parquet


,title,subreddit,engagement_prob,safety_prob
0,"People who haven't pooped in 2019 yet, why are...",askreddit,0.928767,0.370059
1,How would you feel about Reddit adding 3 NSFW ...,askreddit,0.834848,0.933638
2,Would you watch a show where a billionaire CEO...,askreddit,0.962924,0.136665
3,"What if God came down one day and said ""It's p...",askreddit,0.816344,0.076051
4,How would you feel about a feature where if so...,askreddit,0.824065,0.104165


In [3]:
# --- Define user assumptions ---

# Marketing
campaign_budget = 1000                  # Total promotion budget
cost_per_promotion = 10                 # Cost to promote one post
value_per_successful_promotion = 25     # Estimated value of a successful promotion

# Operations
max_moderation_budget = 500             # Maximum budget allocated for moderation
cost_per_moderation = 5                 # Cost to moderate one post

# Safety thresholds
review_threshold = 0.7                    # Threshold for content moderation review
exclude_threshold = 0.9                   # Threshold for excluding content from promotion

In [4]:
# Apply safety thresholds
def apply_safety_thresholds(row):
    if row["safety_prob"] >= exclude_threshold:
        return "Excluded"
    elif row["safety_prob"] >= review_threshold:
        return "Under Review"
    else:
        return "Safe"

decision_system_df = decision_system_df.copy()
decision_system_df["safety_bucket"] = decision_system_df.apply(apply_safety_thresholds, axis=1)

decision_system_df.head()

,title,subreddit,engagement_prob,safety_prob,safety_bucket
0,"People who haven't pooped in 2019 yet, why are...",askreddit,0.928767,0.370059,Safe
1,How would you feel about Reddit adding 3 NSFW ...,askreddit,0.834848,0.933638,Excluded
2,Would you watch a show where a billionaire CEO...,askreddit,0.962924,0.136665,Safe
3,"What if God came down one day and said ""It's p...",askreddit,0.816344,0.076051,Safe
4,How would you feel about a feature where if so...,askreddit,0.824065,0.104165,Safe


In [5]:
# Split content into moderation buckets
excluded_content = decision_system_df[decision_system_df["safety_bucket"] == "Excluded"].copy()
moderation_content = decision_system_df[decision_system_df["safety_bucket"] == "Under Review"].copy()
safe_content = decision_system_df[decision_system_df["safety_bucket"] == "Safe"].copy()

In [6]:
# --- Apply moderation capacity constraints ---

# Calculate how many posts can be moderated within the budget
max_review_capacity = int(max_moderation_budget // cost_per_moderation)

# Prioritise highest risk content for review
to_review = (moderation_content
    .sort_values(by="safety_prob", ascending=False)
    .head(max_review_capacity)
    .copy()
)

# Maintain a record of content that was excluded due to moderation capacity limits
unmoderated_content = moderation_content.drop(to_review.index).copy()

# Calculate moderation costs
total_moderation_cost = len(to_review) * cost_per_moderation
print(f"Total moderation cost: ${total_moderation_cost} for {len(to_review)} posts")

Total moderation cost: $500 for 100 posts


In [8]:
# --- Decide promotion strategy ---

# Calculate promotion slots available
promotion_slots = int(campaign_budget // cost_per_promotion)

# Cannot promote more than the number of safe posts available
promotion_slots = min(promotion_slots, len(safe_content))

# Select content to promote (prioritise highest value content)
to_promote = (safe_content
    .sort_values(by="engagement_prob", ascending=False)
    .head(promotion_slots)
    .copy()
)

# Maintain a record of content that was not promoted due to budget limits
unpromoted_content = safe_content.drop(to_promote.index).copy()

# Calculate promotion costs
total_promotion_cost = len(to_promote) * cost_per_promotion
print(f"Total promotion cost: ${total_promotion_cost} for {len(to_promote)} posts")

Total promotion cost: $1000 for 100 posts


In [ ]:
# --- Estimate value and ROI ---

# Estimate expected value of promotions based on predicted engagement probabilities
expected_success = to_promote["engagement_prob"].sum()
expected_value = expected_success * value_per_successful_promotion

# Calculate ROI
total_cost = total_promotion_cost + total_moderation_cost
net_value = expected_value - total_cost
roi = (net_value / total_cost) * 100 if total_cost > 0 else 0

# Sense check value via effective engagement threshold
effective_engagement_threshold = (
    to_promote["engagement_prob"].min()
    if len(to_promote) > 0
    else None
)

print(f"Expected value: ${expected_value}")
print(f"ROI: {roi:.2f}%")
print(f"Effective engagement threshold for promotion: {effective_engagement_threshold:.2f}")


Expected value: $2432.80738765357
ROI: 62.19%
Effective engagement threshold for promotion: 0.96


In [12]:
# Print summary in full

print("CONTENT SAFETY")
print("Safe:", len(safe_content))
print("Review candidates:", len(moderation_content))
print("Reviewed:", len(to_review))
print("Unreviewed / held back:", len(unmoderated_content))
print("Excluded:", len(excluded_content))

print("\nBUDGETS")
print("Promotion budget:", round(campaign_budget, 2))
print("Moderation budget:", round(max_moderation_budget, 2))
print("Promotion cost used:", round(total_promotion_cost, 2))
print("Moderation cost used:", round(total_moderation_cost, 2))
print("Unused promotion budget:", round(campaign_budget - total_promotion_cost, 2))
print("Unused moderation budget:", round(max_moderation_budget - total_moderation_cost, 2))

print("\nPROMOTION")
print("Promotion slots:", promotion_slots)
print("Promoted:", len(to_promote))
print("Not promoted but safe:", len(unpromoted_content))

if effective_engagement_threshold is not None:
    print("Effective engagement threshold:", round(effective_engagement_threshold, 3))
else:
    print("Effective engagement threshold: None")

print("\nEXPECTED VALUE")
print("Expected successful posts:", round(expected_success, 1))
print("Expected value:", round(expected_value, 2))
print("Total cost:", round(total_cost, 2))
print("Net value:", round(net_value, 2))
print(f"ROI: {roi:.1f}%")

CONTENT SAFETY
Safe: 6097
Review candidates: 533
Reviewed: 100
Unreviewed / held back: 433
Excluded: 306

BUDGETS
Promotion budget: 1000
Moderation budget: 500
Promotion cost used: 1000
Moderation cost used: 500
Unused promotion budget: 0
Unused moderation budget: 0

PROMOTION
Promotion slots: 100
Promoted: 100
Not promoted but safe: 5997
Effective engagement threshold: 0.964

EXPECTED VALUE
Expected successful posts: 97.3
Expected value: 2432.81
Total cost: 1500
Net value: 932.81
ROI: 62.2%


With these assumptions, the system promotes 100 posts, expects ~97 high-performing posts, and generates an estimated net value of £932.81 after promotion and moderation costs - a **62.2% return** on total spend.

In a production setting, new content would be processed through the same feature pipeline and models to generate updated engagement and safety scores. These are combined into a decision dataset, which feeds downstream systems such as ranking, moderation workflows, or planning tools.